In [3]:
import pandas as pd
import numpy as np
# Gerando um dataset de exemplo de voos com algumas colunas básicas
data = {
    'AIRLINE': np.random.choice(['AA', 'DL', 'UA', 'SW', 'FL'], size=100),
    'ORIGIN_AIRPORT': np.random.choice(['JFK', 'LAX', 'ORD', 'ATL', 'SFO'], size=100),
    'DESTINATION_AIRPORT': np.random.choice(['JFK', 'LAX', 'ORD', 'ATL', 'SFO'], size=100),
    'DEPARTURE_DELAY': np.random.randint(-30, 120, size=100),  # Atraso na partida (em minutos)
    'ARRIVAL_DELAY': np.random.randint(-30, 150, size=100),    # Atraso na chegada (em minutos)
}

# Criando o DataFrame
df = pd.DataFrame(data)

# Salvando o DataFrame como um arquivo .csv
df.to_csv('flights.csv', index=False)

In [4]:
# Importando as bibliotecas necessárias
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, desc

# Criando uma SparkSession (ponto de entrada para usar PySpark)
spark = SparkSession.builder \
    .appName("Flight Data Analysis") \
    .getOrCreate()

# Lendo o dataset CSV (você pode usar um caminho local ou baixar um dataset online)
# Exemplo: dados fictícios de voos
df = spark.read.csv("flights.csv", header=True, inferSchema=True)

# Mostrando as primeiras linhas para entender o formato dos dados
df.show(5)

# Exibindo o esquema (tipos de dados de cada coluna)
df.printSchema()


+-------+--------------+-------------------+---------------+-------------+
|AIRLINE|ORIGIN_AIRPORT|DESTINATION_AIRPORT|DEPARTURE_DELAY|ARRIVAL_DELAY|
+-------+--------------+-------------------+---------------+-------------+
|     DL|           ORD|                LAX|             27|           66|
|     SW|           ORD|                ORD|             26|           57|
|     AA|           SFO|                ORD|            -12|          111|
|     UA|           ATL|                ATL|             17|          -11|
|     FL|           ATL|                LAX|             20|           47|
+-------+--------------+-------------------+---------------+-------------+
only showing top 5 rows

root
 |-- AIRLINE: string (nullable = true)
 |-- ORIGIN_AIRPORT: string (nullable = true)
 |-- DESTINATION_AIRPORT: string (nullable = true)
 |-- DEPARTURE_DELAY: integer (nullable = true)
 |-- ARRIVAL_DELAY: integer (nullable = true)



In [5]:
# Selecionando apenas as colunas úteis
df_clean = df.select("AIRLINE", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "DEPARTURE_DELAY", "ARRIVAL_DELAY")

# Convertendo atrasos para números e removendo valores nulos
df_clean = df_clean.na.drop(subset=["DEPARTURE_DELAY", "ARRIVAL_DELAY"])


In [6]:
# Top 10 companhias com mais voos
df.groupBy("AIRLINE").count().orderBy(desc("count")).show(10)

# Atraso médio por companhia
df.groupBy("AIRLINE") \
  .agg(avg("ARRIVAL_DELAY").alias("avg_arrival_delay")) \
  .orderBy(desc("avg_arrival_delay")) \
  .show(10)

# Aeroportos com maior média de atraso na chegada
df.groupBy("DESTINATION_AIRPORT") \
  .agg(avg("ARRIVAL_DELAY").alias("avg_arrival_delay")) \
  .orderBy(desc("avg_arrival_delay")) \
  .show(10)


+-------+-----+
|AIRLINE|count|
+-------+-----+
|     UA|   25|
|     AA|   22|
|     FL|   22|
|     DL|   16|
|     SW|   15|
+-------+-----+

+-------+-----------------+
|AIRLINE|avg_arrival_delay|
+-------+-----------------+
|     UA|            82.08|
|     DL|            62.25|
|     SW|53.46666666666667|
|     AA|51.86363636363637|
|     FL|             50.0|
+-------+-----------------+

+-------------------+------------------+
|DESTINATION_AIRPORT| avg_arrival_delay|
+-------------------+------------------+
|                ATL| 66.38461538461539|
|                SFO| 65.72222222222223|
|                ORD|  65.6842105263158|
|                LAX|58.172413793103445|
|                JFK|52.857142857142854|
+-------------------+------------------+



In [7]:
# Encerrar a sessão do Spark
spark.stop()
